<a href="https://colab.research.google.com/github/vanashri-18/CSA6101-Digital-Forensics-and-Cybercrime-Investigation/blob/main/Authentication_Event_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Aim**

To develop a Python parser for simulated Windows Event Log records that identifies repeated failed authentication attempts, groups them by user account and source IP address, and detects accounts or sources that exceed an investigator-defined failure threshold.

**Algorithm**

Create or load simulated Windows authentication event records.

Read the timestamp, username, source IP, event ID, and authentication status.

Identify failed authentication events.

Ask the investigator to specify a suspicious failure threshold.

Group failed events by username.

Group failed events by source IP address.

Count the number of failures in each group.

Compare each count with the selected threshold.

Identify accounts and source addresses that cross the threshold.

Display the suspicious accounts, sources, and supporting event details.


In [1]:
# ==============================================
# WINDOWS AUTHENTICATION EVENT ANALYZER
# ==============================================

import pandas as pd

# ------------------------------------------------
# 1. Simulated Windows Event Log
# ------------------------------------------------

data = [
    ["2026-08-25 08:10:00", 4625, "alice", "192.168.1.10", "Failed"],
    ["2026-08-25 08:11:00", 4625, "alice", "192.168.1.10", "Failed"],
    ["2026-08-25 08:12:00", 4625, "alice", "192.168.1.10", "Failed"],
    ["2026-08-25 08:13:00", 4625, "alice", "192.168.1.10", "Failed"],
    ["2026-08-25 08:15:00", 4624, "alice", "192.168.1.10", "Successful"],

    ["2026-08-25 08:20:00", 4625, "bob", "192.168.1.20", "Failed"],
    ["2026-08-25 08:21:00", 4625, "bob", "192.168.1.20", "Failed"],

    ["2026-08-25 08:30:00", 4625, "charlie", "10.0.0.5", "Failed"],
    ["2026-08-25 08:31:00", 4625, "charlie", "10.0.0.5", "Failed"],
    ["2026-08-25 08:32:00", 4625, "charlie", "10.0.0.5", "Failed"],
    ["2026-08-25 08:33:00", 4625, "charlie", "10.0.0.5", "Failed"],

    ["2026-08-25 08:40:00", 4624, "david", "10.0.0.8", "Successful"],
    ["2026-08-25 08:45:00", 4625, "david", "10.0.0.8", "Failed"]
]

df = pd.DataFrame(
    data,
    columns=[
        "Timestamp",
        "Event_ID",
        "Username",
        "Source_IP",
        "Status"
    ]
)

df["Timestamp"] = pd.to_datetime(
    df["Timestamp"]
)

print("=" * 80)
print("             WINDOWS AUTHENTICATION EVENT ANALYZER")
print("=" * 80)

# ------------------------------------------------
# 2. Investigator Threshold
# ------------------------------------------------

threshold = int(
    input(
        "\nEnter suspicious failure threshold: "
    )
)

# ------------------------------------------------
# 3. Extract Failed Authentication Events
# ------------------------------------------------

failed = df[
    (df["Event_ID"] == 4625) &
    (df["Status"] == "Failed")
].copy()

print("\nTotal Failed Authentication Events:",
      len(failed))

# ------------------------------------------------
# 4. Group by Username
# ------------------------------------------------

user_counts = (
    failed
    .groupby("Username")
    .size()
    .reset_index(name="Failure_Count")
)

suspicious_users = user_counts[
    user_counts["Failure_Count"] >= threshold
]

# ------------------------------------------------
# 5. Group by Source IP
# ------------------------------------------------

ip_counts = (
    failed
    .groupby("Source_IP")
    .size()
    .reset_index(name="Failure_Count")
)

suspicious_ips = ip_counts[
    ip_counts["Failure_Count"] >= threshold
]

# ------------------------------------------------
# 6. Display Suspicious Accounts
# ------------------------------------------------

print("\n" + "=" * 80)
print("              SUSPICIOUS USER ACCOUNTS")
print("=" * 80)

if suspicious_users.empty:

    print("No user account crossed the threshold.")

else:

    print(
        suspicious_users.to_string(
            index=False
        )
    )

# ------------------------------------------------
# 7. Display Suspicious Source Addresses
# ------------------------------------------------

print("\n" + "=" * 80)
print("              SUSPICIOUS SOURCE ADDRESSES")
print("=" * 80)

if suspicious_ips.empty:

    print("No source address crossed the threshold.")

else:

    print(
        suspicious_ips.to_string(
            index=False
        )
    )

# ------------------------------------------------
# 8. Supporting Events
# ------------------------------------------------

print("\n" + "=" * 80)
print("                 SUPPORTING EVENTS")
print("=" * 80)

if not suspicious_users.empty:

    users = suspicious_users["Username"].tolist()

    user_events = failed[
        failed["Username"].isin(users)
    ]

    print("\nEvents associated with suspicious accounts:")
    print(
        user_events.to_string(index=False)
    )

# ------------------------------------------------
# 9. Summary
# ------------------------------------------------

print("\n" + "=" * 80)
print("                       SUMMARY")
print("=" * 80)

print("Failure Threshold :", threshold)
print("Failed Events     :", len(failed))
print(
    "Suspicious Users  :",
    len(suspicious_users)
)
print(
    "Suspicious IPs    :",
    len(suspicious_ips)
)

print("\nAnalysis completed.")
print("=" * 80)

             WINDOWS AUTHENTICATION EVENT ANALYZER

Enter suspicious failure threshold: 3

Total Failed Authentication Events: 11

              SUSPICIOUS USER ACCOUNTS
Username  Failure_Count
   alice              4
 charlie              4

              SUSPICIOUS SOURCE ADDRESSES
   Source_IP  Failure_Count
    10.0.0.5              4
192.168.1.10              4

                 SUPPORTING EVENTS

Events associated with suspicious accounts:
          Timestamp  Event_ID Username    Source_IP Status
2026-08-25 08:10:00      4625    alice 192.168.1.10 Failed
2026-08-25 08:11:00      4625    alice 192.168.1.10 Failed
2026-08-25 08:12:00      4625    alice 192.168.1.10 Failed
2026-08-25 08:13:00      4625    alice 192.168.1.10 Failed
2026-08-25 08:30:00      4625  charlie     10.0.0.5 Failed
2026-08-25 08:31:00      4625  charlie     10.0.0.5 Failed
2026-08-25 08:32:00      4625  charlie     10.0.0.5 Failed
2026-08-25 08:33:00      4625  charlie     10.0.0.5 Failed

                  

**Result**

The Python program successfully parsed simulated Windows authentication events and isolated failed login events (Event ID 4625) from successful authentication events. Failed attempts were grouped by username and source IP address, and groups meeting or exceeding the investigator-defined threshold were identified as suspicious for further investigation. The program also displayed supporting events so that the investigator can verify the evidence behind each detection.